# Viscan CBCT Bonus - Scan #2: PreDentalSurgery
Runs the **pretrained, as-is** DentalSegmentator (Dot et al. 2024, Zenodo 10.5281/zenodo.10829675, CC BY 4.0) on the *PreDentalSurgery* CBCT (3D Slicer SampleData) and packages `PreDentalSurgery_bundle.zip` (volume + segmentation).
No reference mask exists for this scan, so **no Dice is claimed**.

In [ ]:
!pip install -q "nnunetv2>=2.2.1,<2.5.0" SimpleITK nibabel psutil
import os, time, json, hashlib, zipfile, urllib.request
from pathlib import Path
import numpy as np, torch, SimpleITK as sitk, psutil
WORK = Path('/kaggle/working/cbct_pre'); WORK.mkdir(parents=True, exist_ok=True)
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# Weights (verified)
URL='https://zenodo.org/api/records/10829675/files/Dataset112_DentalSegmentator_v100.zip/content'
EXP_MD5='b71cd5230168d28a4f71b078265b76be'; EXP_BYTES=229747861
W=WORK/'weights'; W.mkdir(exist_ok=True); z=W/'w.zip'
if not z.exists() or z.stat().st_size!=EXP_BYTES: urllib.request.urlretrieve(URL, z)
assert z.stat().st_size==EXP_BYTES and hashlib.md5(z.read_bytes()).hexdigest()==EXP_MD5
ex=W/'extracted'
if not ex.exists(): zipfile.ZipFile(z).extractall(ex)
MODEL=next(ex.rglob('dataset.json')).parent; print(MODEL)

In [ ]:
# PreDentalSurgery volume (SHA256 verified, 3D Slicer SlicerTestingData)
SHA='7bfa16945629c319a439f414cfb7edddd2a97ba97753e12eede3b56a0eb09968'
g=WORK/'Pre.gipl.gz'
if not g.exists(): urllib.request.urlretrieve('https://github.com/Slicer/SlicerTestingData/releases/download/SHA256/'+SHA, g)
assert hashlib.sha256(g.read_bytes()).hexdigest()==SHA
img=sitk.ReadImage(str(g)); print(img.GetSize(), img.GetSpacing())
IN=WORK/'in'; OUT=WORK/'out'; IN.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
sitk.WriteImage(img, str(IN/'PreDentalSurgery_0000.nii.gz'))

In [ ]:
# Inference (pretrained as-is; TTA off, same settings as the Post run)
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'
os.environ['nnUNet_raw']=os.environ['nnUNet_preprocessed']=os.environ['nnUNet_results']=str(W)
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
dev=torch.device('cuda',0) if torch.cuda.is_available() else torch.device('cpu')
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
p=nnUNetPredictor(tile_step_size=0.5,use_gaussian=True,use_mirroring=False,perform_everything_on_device=torch.cuda.is_available(),device=dev,verbose=False,allow_tqdm=True)
p.initialize_from_trained_model_folder(str(MODEL),use_folds=(0,),checkpoint_name='checkpoint_final.pth')
t=time.perf_counter()
p.predict_from_files([[str(IN/'PreDentalSurgery_0000.nii.gz')]],[str(OUT/'PreDentalSurgery.nii.gz')],save_probabilities=False,overwrite=True,num_processes_preprocessing=1,num_processes_segmentation_export=1)
ms=(time.perf_counter()-t)*1000
vram=torch.cuda.max_memory_allocated()/2**20 if torch.cuda.is_available() else 0
print('inference_ms',round(ms,1),'peak_vram_mb',round(vram,1),'rss_mb',round(psutil.Process().memory_info().rss/2**20,1))
seg=sitk.GetArrayFromImage(sitk.ReadImage(str(OUT/'PreDentalSurgery.nii.gz')))
print({int(c):int((seg==c).sum()) for c in range(6)})

In [ ]:
# Bundle: volume + segmentation (+ run metrics). 'seg' in the filename is what the app uses to identify the label map.
B=Path('/kaggle/working/PreDentalSurgery_bundle.zip')
with zipfile.ZipFile(B,'w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(IN/'PreDentalSurgery_0000.nii.gz', 'PreDentalSurgery_Volume.nii.gz')
    zf.write(OUT/'PreDentalSurgery.nii.gz', 'PreDentalSurgery_Segmentation.nii.gz')
    zf.writestr('run_metrics.json', json.dumps({'inference_ms':round(ms,1),'peak_vram_mb':round(vram,1)}))
print(B, round(B.stat().st_size/2**20,1),'MB')